### 명확한 인스트럭션
- bad
    - 감정을 찾아줘
- good
    - 다음 영화 리뷰의 감정을 긍정/부정/중립 중 하나로 분류하세요 응답은 감정 단어 하나만 출력하세요

### 일관된 포멧
- 리뷰1 : ...
- 감정 : 긍정

- 리뷰2 : ...
- 감정 : 부정

- 리뷰 : ...
- 감정 : 긍정

- 리뷰 : ...
- 감정 : 부정

### 다양한 예시 : 균형잡히게
- 예시 1 : "좋아" ->긍정
- 예시 1 : "최고야" ->긍정
- 예시 1 : "훌륭해" ->긍정

- 예시 1 : "좋아" ->긍정
- 예시 1 : "싫어" ->부정
- 예시 1 : "그럭저럭" ->중립

### 예시의 순서효과
- 긍정1, 긍정2, 부정, 중립 -> 모델이 긍정을 과도하게 예측 과적합 상태

- 긍정,부정,중립 -> 균형있게 학습

### 추천
- 클래스별 균형
- 어려운 경우부터 시작(모델이 주의집중)
- 같은 클래스 반복 피하기
- 무작위 섞기보다 신중하게 배열

### 전략
- 다양성 극대화
    - 선택기준:
        - 각 클래스를 대표하는 명확한 사례
        - 경계 사례 포함(모호한 케이스)
        - 도메일 특화 어휘 포함
- 어려움 레벨 고려
    - Easy : '이 영화 최고야! (명확한 긍정)
    - Midium : '그런대로 나쁘지 않네'(부정 + 약한표현)
    - Hard : .... (도메인특화, 뉘양스 중요)

    - 프롬프트 : Easy->Medium->Hard 순서


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import permutations

In [2]:
# 법률 문서 조항 데이터
legal_data = [
    {
        "text": "본 계약의 유효 기간은 2024년 1월 1일부터 2025년 12월 31일까지이다.",
        "category": "기간",
        "difficulty": "easy"
    },
    {
        "text": "계약 당사자가 본 계약의 어떤 조항을 위반할 경우 상대방은 즉시 계약을 해제할 수 있다.",
        "category": "위약금",
        "difficulty": "medium"
    },
    {
        "text": "갑은 본 서비스와 관련하여 발생한 모든 손해에 대해 책임을 지지 아니한다.",
        "category": "책임제한",
        "difficulty": "medium"
    },
    {
        "text": "본 계약에 명시되지 않은 사항에 대해서는 관계 법령을 준용한다.",
        "category": "준거법",
        "difficulty": "hard"
    },
    {
        "text": "이용자는 언제든지 7일 이전 통지로 본 계약을 해지할 수 있다.",
        "category": "해지",
        "difficulty": "easy"
    },
    {
        "text": "당사자 간의 분쟁이 발생할 경우 서울중앙지방법원을 관할 법원으로 한다.",
        "category": "관할법원",
        "difficulty": "hard"
    },
]

test_data = [
    {"text": "본 서비스는 2024년 1월부터 시작된다.", "category": "기간"},
    {"text": "계약 위반 시 연 10%의 위약금을 청구할 수 있다.", "category": "위약금"},
    {"text": "본사는 서비스 중단에 대해 책임지지 않는다.", "category": "책임제한"},
    {"text": "분쟁은 중재법에 따라 처리된다.", "category": "준거법"},
]

print(f"훈련 데이터: {len(legal_data)}개")
for i, d in enumerate(legal_data, 1):
    print(f"  {i}. [{d['category']:8}] {d['text'][:40]}...")

print(f"\n테스트 데이터: {len(test_data)}개")
for i, d in enumerate(test_data, 1):
    print(f"  {i}. {d['text'][:50]}...")

훈련 데이터: 6개
  1. [기간      ] 본 계약의 유효 기간은 2024년 1월 1일부터 2025년 12월 31일...
  2. [위약금     ] 계약 당사자가 본 계약의 어떤 조항을 위반할 경우 상대방은 즉시 계약을 ...
  3. [책임제한    ] 갑은 본 서비스와 관련하여 발생한 모든 손해에 대해 책임을 지지 아니한다...
  4. [준거법     ] 본 계약에 명시되지 않은 사항에 대해서는 관계 법령을 준용한다....
  5. [해지      ] 이용자는 언제든지 7일 이전 통지로 본 계약을 해지할 수 있다....
  6. [관할법원    ] 당사자 간의 분쟁이 발생할 경우 서울중앙지방법원을 관할 법원으로 한다....

테스트 데이터: 4개
  1. 본 서비스는 2024년 1월부터 시작된다....
  2. 계약 위반 시 연 10%의 위약금을 청구할 수 있다....
  3. 본사는 서비스 중단에 대해 책임지지 않는다....
  4. 분쟁은 중재법에 따라 처리된다....


In [3]:
class PromptOptimizer:
    """프롬프트 최적화 클래스"""
    
    def __init__(self):
        self.category_map = {
            "기간": ["시간", "기간", "날짜", "년도", "일자"],
            "위약금": ["위약금", "손해배상", "페널티", "과태료", "위반"],
            "책임제한": ["책임", "담당", "책임제한", "면책", "책임지지"],
            "준거법": ["법령", "준거법", "법률", "규정", "법"],
            "해지": ["해지", "종료", "종료", "폐기", "중단"],
            "관할법원": ["법원", "중재", "소송", "분쟁", "관할"],
        }
    
    def classify_simple(self, text):
        """키워드 기반 분류"""
        text_lower = text.lower()
        scores = {}
        
        for category, keywords in self.category_map.items():
            score = sum(1 for kw in keywords if kw in text_lower)
            scores[category] = score
        
        if max(scores.values()) == 0:
            return "기타"
        
        return max(scores, key=scores.get)

optimizer = PromptOptimizer()
print("프롬프트 최적화 클래스 초기화 완료")

프롬프트 최적화 클래스 초기화 완료


In [4]:
def create_prompt(examples, order="original"):
    """프롬프트 생성"""
    
    if order == "random":
        examples = list(examples)
        np.random.shuffle(examples)
    elif order == "difficulty":
        # 어려움 순서: easy → medium → hard
        difficulty_order = {"easy": 0, "medium": 1, "hard": 2}
        examples = sorted(examples, key=lambda x: difficulty_order.get(x.get("difficulty", "easy"), 0))
    elif order == "balanced":
        # 각 카테고리가 고르게 분포
        by_category = {}
        for ex in examples:
            cat = ex["category"]
            if cat not in by_category:
                by_category[cat] = []
            by_category[cat].append(ex)
        
        examples = []
        max_len = max(len(v) for v in by_category.values())
        for i in range(max_len):
            for cat in by_category:
                if i < len(by_category[cat]):
                    examples.append(by_category[cat][i])
    
    prompt = """다음 법률 문서의 조항을 카테고리로 분류하세요.
카테고리: 기간, 위약금, 책임제한, 준거법, 해지, 관할법원
응답 형식: 카테고리명만 출력하세요.

"""
    
    for ex in examples:
        prompt += f"조항: {ex['text']}\n"
        prompt += f"카테고리: {ex['category']}\n\n"
    
    return prompt, examples

# 4가지 순서로 프롬프트 생성
orders = ["original", "random", "difficulty", "balanced"]

for order in orders:
    prompt, ex_order = create_prompt(legal_data[:3], order=order)
    print(f"\n{'='*60}")
    print(f"[순서: {order.upper()}]")
    print(f"{'='*60}")
    print(prompt[:300] + "...")
    print(f"예시 순서: {[ex['category'] for ex in ex_order]}")


[순서: ORIGINAL]
다음 법률 문서의 조항을 카테고리로 분류하세요.
카테고리: 기간, 위약금, 책임제한, 준거법, 해지, 관할법원
응답 형식: 카테고리명만 출력하세요.

조항: 본 계약의 유효 기간은 2024년 1월 1일부터 2025년 12월 31일까지이다.
카테고리: 기간

조항: 계약 당사자가 본 계약의 어떤 조항을 위반할 경우 상대방은 즉시 계약을 해제할 수 있다.
카테고리: 위약금

조항: 갑은 본 서비스와 관련하여 발생한 모든 손해에 대해 책임을 지지 아니한다.
카테고리: 책임제한

...
예시 순서: ['기간', '위약금', '책임제한']

[순서: RANDOM]
다음 법률 문서의 조항을 카테고리로 분류하세요.
카테고리: 기간, 위약금, 책임제한, 준거법, 해지, 관할법원
응답 형식: 카테고리명만 출력하세요.

조항: 갑은 본 서비스와 관련하여 발생한 모든 손해에 대해 책임을 지지 아니한다.
카테고리: 책임제한

조항: 본 계약의 유효 기간은 2024년 1월 1일부터 2025년 12월 31일까지이다.
카테고리: 기간

조항: 계약 당사자가 본 계약의 어떤 조항을 위반할 경우 상대방은 즉시 계약을 해제할 수 있다.
카테고리: 위약금

...
예시 순서: ['책임제한', '기간', '위약금']

[순서: DIFFICULTY]
다음 법률 문서의 조항을 카테고리로 분류하세요.
카테고리: 기간, 위약금, 책임제한, 준거법, 해지, 관할법원
응답 형식: 카테고리명만 출력하세요.

조항: 본 계약의 유효 기간은 2024년 1월 1일부터 2025년 12월 31일까지이다.
카테고리: 기간

조항: 계약 당사자가 본 계약의 어떤 조항을 위반할 경우 상대방은 즉시 계약을 해제할 수 있다.
카테고리: 위약금

조항: 갑은 본 서비스와 관련하여 발생한 모든 손해에 대해 책임을 지지 아니한다.
카테고리: 책임제한

...
예시 순서: ['기간', '위약금', '책임제한']

[순서: BALANCED]
다음 법률 문서의 조항을 카테고리로 분류하세요.
카테고리: 기간,

In [5]:
def evaluate_prompt(examples, test_data, optimizer):
    """프롬프트 평가"""
    
    # 테스트 데이터 분류
    predictions = []
    for test in test_data:
        pred = optimizer.classify_simple(test['text'])
        predictions.append(pred)
    
    # 정확도 계산
    correct = sum(1 for pred, test in zip(predictions, test_data) if pred == test['category'])
    accuracy = correct / len(test_data)
    
    return accuracy, predictions

# 각 순서별 성능 평가
results = []

for order in orders:
    prompt, ex_order = create_prompt(legal_data[:4], order=order)
    accuracy, preds = evaluate_prompt(ex_order, test_data, optimizer)
    
    results.append({
        '순서': order.upper(),
        '예시배열': ' → '.join([ex['category'] for ex in ex_order]),
        '정확도': f'{accuracy:.0%}',
        '정확한수': f'{int(accuracy * len(test_data))}/{len(test_data)}'
    })

results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("[프롬프트 순서 효과 분석]")
print("="*80)
print(results_df.to_string(index=False))


[프롬프트 순서 효과 분석]
        순서                  예시배열 정확도 정확한수
  ORIGINAL 기간 → 위약금 → 책임제한 → 준거법 50%  2/4
    RANDOM 준거법 → 위약금 → 기간 → 책임제한 50%  2/4
DIFFICULTY 기간 → 위약금 → 책임제한 → 준거법 50%  2/4
  BALANCED 기간 → 위약금 → 책임제한 → 준거법 50%  2/4


In [6]:
def create_optimized_prompt(examples):
    """최적화된 프롬프트 생성"""
    
    prompt = """당신은 법률 문서 분석 전문가입니다.
다음 법률 조항의 카테고리를 정확히 분류하세요.

## 카테고리 정의
- 기간: 계약의 유효 기간, 시작/종료 날짜
- 위약금: 계약 위반 시 페널티, 손해배상
- 책임제한: 당사자의 책임 범위 제한, 면책 조항
- 준거법: 계약 분쟁 시 적용 법령, 법률 준용
- 해지: 계약 해지 조건, 해지 절차
- 관할법원: 분쟁 해결 절차, 소송 관할권

## 분류 예시
"""
    
    for ex in examples:
        prompt += f"조항: \"{ex['text']}\"\n"
        prompt += f"→ 카테고리: {ex['category']}\n\n"
    
    prompt += "## 분류할 조항\n"
    prompt += "조항: \"{text}\"\n"
    prompt += "→ 카테고리: "
    
    return prompt

# 최적 프롬프트 생성
optimized_prompt = create_optimized_prompt(legal_data[:3])
print("\n" + "="*60)
print("[최적화된 프롬프트 템플릿]")
print("="*60)
print(optimized_prompt)

print("\n\n최적화 포인트:")
print("  1. 명확한 카테고리 정의 제공")
print("  2. 일관된 포맷 사용")
print("  3. 다양한 예시 포함")
print("  4. 이해하기 쉬운 언어")
print("  5. 출력 형식 명시")


[최적화된 프롬프트 템플릿]
당신은 법률 문서 분석 전문가입니다.
다음 법률 조항의 카테고리를 정확히 분류하세요.

## 카테고리 정의
- 기간: 계약의 유효 기간, 시작/종료 날짜
- 위약금: 계약 위반 시 페널티, 손해배상
- 책임제한: 당사자의 책임 범위 제한, 면책 조항
- 준거법: 계약 분쟁 시 적용 법령, 법률 준용
- 해지: 계약 해지 조건, 해지 절차
- 관할법원: 분쟁 해결 절차, 소송 관할권

## 분류 예시
조항: "본 계약의 유효 기간은 2024년 1월 1일부터 2025년 12월 31일까지이다."
→ 카테고리: 기간

조항: "계약 당사자가 본 계약의 어떤 조항을 위반할 경우 상대방은 즉시 계약을 해제할 수 있다."
→ 카테고리: 위약금

조항: "갑은 본 서비스와 관련하여 발생한 모든 손해에 대해 책임을 지지 아니한다."
→ 카테고리: 책임제한

## 분류할 조항
조항: "{text}"
→ 카테고리: 


최적화 포인트:
  1. 명확한 카테고리 정의 제공
  2. 일관된 포맷 사용
  3. 다양한 예시 포함
  4. 이해하기 쉬운 언어
  5. 출력 형식 명시


In [7]:
# 허깅페이스 모델로 최적화 프롬프트 실행
# 입력 : 최적화된 프롬프트 + 테스트 조항
# 출력 : 기간 위약금 책임제한 준거법 해지 관할법원 중 하나

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
).to(device)
categories = ['기간', '위약금', '책임제한', '준거법', '해지', '관할법원']

def normalize_category(text):
    """모델 출력에서 카테고리명만 추출"""
    for category in categories:
        if category in text:
            return category
    return text.strip()

def classify_with_huggingface(clause):
    prompt = optimized_prompt.format(text=clause)
    messages = [
        {
            "role": "system",
            "content": "너는 법률 문서 조항을 정해진 카테고리 중 하나로만 분류하는 전문가야. 설명 없이 카테고리명만 출력해."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    chat_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(chat_prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    raw_output = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return normalize_category(raw_output), raw_output

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [10]:
test_data

[{'text': '본 서비스는 2024년 1월부터 시작된다.', 'category': '기간'},
 {'text': '계약 위반 시 연 10%의 위약금을 청구할 수 있다.', 'category': '위약금'},
 {'text': '본사는 서비스 중단에 대해 책임지지 않는다.', 'category': '책임제한'},
 {'text': '분쟁은 중재법에 따라 처리된다.', 'category': '준거법'}]

In [ ]:
predictions = []
for item in test_data:
    prediction, raw_output =  classify_with_huggingface(item['text'])
    predictions.append({
        '조항' : item['text'],
        '정답' : item['category'],
        '예측' : prediction,
        '원본출력':raw_output,
        '정답여부' : prediction == item['category']
    })
result_df = pd.DataFrame(predictions)
accuracy = result_df['정답여부'].mean()
print('정확도',accuracy)